In [1]:
import os
import scanpy as sc
from scipy.sparse import csr_matrix
import numpy as np

#NEW Diversidad Humana
categories = {
    
    "B cells": [
        "naive B cell",
        "transitional stage B cell",
        "memory B cell",
        "class switched memory B cell",
        "unswitched memory B cell",
        "IgG memory B cell",
        "IgG-negative class switched memory B cell",
        "plasma cell",
        "plasmablast",
        "IgA plasmablast",
        "IgG plasmablast",
        "IgA plasma cell",
        "IgM plasma cell",
        "IgG plasma cell",
        "B cell",
        "mature B cell",
        "immature B cell",
        "precursor B cell",
        "pro-B cell"
    ],
    
    "T cells": [
        "CD4-positive, CD25-positive, alpha-beta regulatory T cell",
        "double negative thymocyte",
        "double-positive, alpha-beta thymocyte",
        "CD4-positive, alpha-beta memory T cell",
        "CD4-positive, alpha-beta T cell",
        "naive thymus-derived CD4-positive, alpha-beta T cell",
        "CD4-positive helper T cell",
        "effector memory CD4-positive, alpha-beta T cell",
        "central memory CD4-positive, alpha-beta T cell",
        "activated CD4-positive, alpha-beta T cell",
        "activated CD4-positive, alpha-beta T cell, human",
        "CD4-positive, alpha-beta cytotoxic T cell",
        "effector CD4-positive, alpha-beta T cell",
        "CD8-positive, alpha-beta T cell",
        "naive thymus-derived CD8-positive, alpha-beta T cell",
        "CD8-positive, alpha-beta cytokine secreting effector T cell",
        "CD8-positive, alpha-beta memory T cell",
        "CD8-positive, alpha-beta memory T cell, CD45RO-positive",
        "central memory CD8-positive, alpha-beta T cell",
        "effector memory CD8-positive, alpha-beta T cell",
        "effector memory CD8-positive, alpha-beta T cell, terminally differentiated",
        "CD8-positive, alpha-beta cytotoxic T cell",
        "activated CD8-positive, alpha-beta T cell",
        "activated CD8-positive, alpha-beta T cell, human",
        "effector CD8-positive, alpha-beta T cell",
        "T cell",
        "alpha-beta T cell",
        "mature alpha-beta T cell",
        "memory T cell",
        "naive T cell",
        "regulatory T cell",
        "T follicular helper cell",
        "double negative T regulatory cell",
        "T-helper 1 cell",
        "T-helper 2 cell",
        "T-helper 17 cell",
        "T-helper 22 cell",
        "mucosal invariant T cell",
        "gamma-delta T cell",
        "mature gamma-delta T cell"
    ],
    
    "NK cells": [
        "natural killer cell",
        "CD16-positive, CD56-dim natural killer cell, human",
        "CD16-negative, CD56-bright natural killer cell, human",
        "mature NK T cell",
        "type I NK T cell",
        "activated type II NK T cell"
    ],
    
    "Myeloid cells": [
        "MHC-II-positive classical monocyte",
        "monocyte",
        "classical monocyte",
        "non-classical monocyte",
        "intermediate monocyte",
        "CD14-positive monocyte",
        "CD14-low, CD16-positive monocyte",
        "CD14-positive, CD16-negative classical monocyte",
        "CD14-positive, CD16-positive monocyte",
        "macrophage",
        "alveolar macrophage",
        "neutrophil",
        "basophil",
        "mast cell",
        "granulocyte",
        "myeloid cell",
        "myeloid leukocyte",
        "late promyelocyte",
        "early promyelocyte",
        "myelocyte",
        "alternatively activated macrophage"
    ],

    "Dendritic cells": [
        "dendritic cell",
        "conventional dendritic cell",
        "CD141-positive myeloid dendritic cell",
        "CD1c-positive myeloid dendritic cell",
        "myeloid dendritic cell",
        "myeloid dendritic cell, human",
        "dendritic cell, human",
        "inflammatory dendritic cell",
        "Langerhans cell"
    ],

    "Dendritic-plasmoid cells": [
        "plasmacytoid dendritic cell",
        "plasmacytoid dendritic cell, human",
    ],
    
    "CD34+ HSPC": [
        "stem cell",
        "hematopoietic stem cell",
        "cord blood hematopoietic stem cell",
        "CD34-positive, CD38-negative hematopoietic stem cell",
        "hematopoietic precursor cell",
        "hematopoietic multipotent progenitor cell",
        "common myeloid progenitor",
        "myeloid lineage restricted progenitor cell",
        "lymphoid lineage restricted progenitor cell",
        "common dendritic progenitor",
        "megakaryocyte-erythroid progenitor cell",
        "megakaryocyte progenitor cell",
        "erythroid progenitor cell",
        "erythroid progenitor cell, mammalian",
        "basophil mast progenitor cell",
        "progenitor cell"
    ],
    
    "Platelets": [
        "platelet",
        "megakaryocyte"
    ],
    
    "RBC": [
        "erythrocyte",
        "erythroid lineage cell",
        "enucleated reticulocyte",
        "erythroblast",
        "proerythroblast"
        
    ],
    
    "Innate Lymphoid Cells": [
        "innate lymphoid cell",
        "group 2 innate lymphoid cell, human",
        "group 3 innate lymphoid cell",
        "ILC1, human",
        "common lymphoid progenitor"
    ],

    "Epithelial cells": [
        "epithelial cell",
        "basal cell",
        "respiratory basal cell",
        "club cell",
        "endothelial cell",
        "epithelial cell of proximal tubule segment 1"
    ],
    
    "Non-specific": [
        "lymphocyte",
        "professional antigen presenting cell",
        "primordial germ cell",
        "blood cell",
        "animal cell",
        "peripheral blood mononuclear cell",
        "unknown",
        "fibroblast", 
        "chondrocyte"
        
    ]
}



In [2]:
# Redefiniendo la función para asignar el grupo de células basado en el tipo de célula y las categorías definidas
def assign_cell_group(cell_type):
    for group, types in categories.items():
        if cell_type in types:
            return group
    return "Uncategorized"  # En caso de que no se encuentre una categoría para un tipo de célula específico


# Function to filter samples
def filtrar_muestra(adata):
    filtro = (
        #(adata.obs['is_primary_data'] == 'True') &
        (adata.obs['tissue'] == 'blood') &
        (adata.obs['disease'] == 'normal') &
        (adata.obs['sex'] != 'unknown') 
        #(adata.obs['organism'] == 'Homo sapiens')
    )
    adata = adata[filtro].copy()
    return adata

# Function to map age stage
def map_age_stage(stage):
    infant_labels = [
        'infant stage', 'newborn human stage', 'adolescent stage',
        '2-5 year-old child stage', '6-12 year-old child stage'
    ]
    adult_labels = [
        'human adult stage', 'third decade human stage',
        'fourth decade human stage'
    ]
    aged_labels = [
        'human aged stage', 'sixth decade human stage',
        'seventh decade human stage', '80 year-old and over human stage',
        'eighth decade human stage', 'fifth decade human stage'
    ]


    if stage in infant_labels:
        return 'young'
    if stage in adult_labels:
        return 'adult'
    if stage in aged_labels:
        return 'old'

    if 'year-old human stage' in stage:
        age = int(stage.split('-')[0])
        if 0 <= age <= 34:
            return 'young'
        elif 34 < age <= 60:
            return 'adult'
        else:
            return 'old'

    return 'other'

# Function to process ages
def procesar_edades(adata):
    edades_explicitas = adata.obs['development_stage'].str.contains('year-old human stage')
    exp_filtrados = adata.obs.loc[edades_explicitas, 'development_stage']
    adata.obs['age_yrs'] = exp_filtrados.str.extract(r'(\d+)').astype(str)
    adata.obs.loc[~edades_explicitas, 'age_yrs'] = 'non_specific'
    adata.obs['age_yrs'] = adata.obs['age_yrs'].astype(str)  # Convertir todos los valores a cadenas
    return adata


# Function to process and save files
def process_and_save_files(directory):
    for filename in os.listdir(directory):
        if filename.endswith(".h5ad"):
            try:
                filepath = os.path.join(directory, filename)
                adata = sc.read_h5ad(filepath)
                # Hacer que los nombres de las variables sean únicos
                adata.var_names_make_unique()
                print(f"\nProcessing {filename}")
                initial_cells = adata.shape[0]
                print(f"Initial shape: {adata.shape}")

                # 1. Procesar datos raw
                if adata.raw is not None and getattr(adata.raw, 'X', None) is not None:
                    adata.X = adata.raw.X
                    del adata.raw
                    print(f"Using raw.X for {filename}")
                    
                    # Verificar si los datos son enteros
                    if isinstance(adata.X, csr_matrix):
                        data = adata.X.data
                    else:
                        data = adata.X

                    if not np.issubdtype(data.dtype, np.integer):
                        print(f"Warning: {filename} - data in adata.X are not integers.")
                    else:
                        print(f"Data in adata.X are integers for {filename}")

                    print(f"First few elements of adata.X after assignment: {data[:5]}")
                else:
                    print(f"raw.X is missing for {filename}, checking if .X contains raw counts.")
                    if not np.all(np.equal(np.mod(adata.X.data if isinstance(adata.X, csr_matrix) else adata.X, 1), 0)):
                        print(f"{filename}: .X does not contain raw counts.")
                        continue

                # 2. Convertir a enteros
                adata.X = adata.X.astype(np.int32)

                # 3. Aplicar filtro de muestra
                adata = filtrar_muestra(adata)
                cells_after_sample_filter = adata.shape[0]
                print(f"Shape after filtrar_muestra: {adata.shape}")
                print(f"Retained {(cells_after_sample_filter/initial_cells)*100:.2f}% of cells after sample filtering")

                # 4. Verificar columna cell_type antes de asignar grupos
                if 'cell_type' not in adata.obs.columns:
                    print(f"Error: 'cell_type' column not found in {filename}")
                    continue

                # 5. Asignar grupos de células
                adata.obs['cell_group'] = adata.obs['cell_type'].apply(assign_cell_group)
                print(f"Applied cell ontology dictionary")

                # 6. Procesar edades
                adata = procesar_edades(adata)
                adata.obs['age_group'] = adata.obs['development_stage'].apply(map_age_stage)
                print(f"Applied age processing")

                
                # 8. Filtrar datos primarios si existe la columna
                if 'is_primary_data' in adata.obs.columns:
                    cells_before_primary = adata.shape[0]
                    adata = adata[adata.obs['is_primary_data'] == True].copy()
                    cells_after_primary = adata.shape[0]
                    print(f"Shape after filtering primary data: {adata.shape}")
                    print(f"Retained {(cells_after_primary/cells_before_primary)*100:.2f}% of cells after primary data filtering")

                # 9. Verificar si hay datos suficientes
                if adata.shape[0] == 0 or adata.shape[1] == 0:
                    print(f"Warning: {filename} is empty or has insufficient data.")
                    continue

                # 10. Manejar valores negativos
                if isinstance(adata.X, csr_matrix):
                    if np.any(adata.X.data < 0):
                        print(f"Negative values found in {filename}, setting them to zero.")
                        adata.X.data = np.maximum(adata.X.data, 0)
                else:
                    if np.any(adata.X < 0):
                        print(f"Negative values found in {filename}, setting them to zero.")
                        adata.X = np.maximum(adata.X, 0)

                # 11. Convertir a matriz dispersa si no lo es
                if not isinstance(adata.X, csr_matrix):
                    adata.X = csr_matrix(adata.X)

                # 12. Filtrar células NK con verificaciones
                cells_before_nk = adata.shape[0]
                if 'cell_group' not in adata.obs.columns:
                    print(f"Error: 'cell_group' column not found after processing {filename}")
                    continue

                # Verificar presencia de células NK
                if 'NK cells' not in adata.obs['cell_group'].unique():
                    print(f"Warning: No NK cells found in {filename}")
                    continue

                # Aplicar filtro NK
                nk_cells_mask = adata.obs['cell_group'] == 'NK cells'
                n_nk_cells = nk_cells_mask.sum()
                
                if n_nk_cells == 0:
                    print(f"Warning: No NK cells found after filtering in {filename}")
                    continue
                    
                adata = adata[nk_cells_mask].copy()
                print(f"""
                NK Cell Filtering Results:
                - Cells before NK filtering: {cells_before_nk}
                - NK cells found: {n_nk_cells}
                - Percentage of NK cells: {(n_nk_cells/cells_before_nk)*100:.2f}%
                """)

                # 13. Guardar archivo procesado
                output_path = os.path.join('/app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final', filename)
                adata.write_h5ad(output_path)
                print(f"Successfully saved processed file as {output_path}")

            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
                continue

# Define the directory containing the .h5ad files
directory = "/app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/"
process_and_save_files(directory)


/root/.pyenv/versions/3.9.7/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



Processing nk_cells.h5ad
Initial shape: (557314, 60530)
raw.X is missing for nk_cells.h5ad, checking if .X contains raw counts.
Shape after filtrar_muestra: (537788, 60530)
Retained 96.50% of cells after sample filtering
Applied cell ontology dictionary
Applied age processing
Shape after filtering primary data: (537788, 60530)
Retained 100.00% of cells after primary data filtering

                NK Cell Filtering Results:
                - Cells before NK filtering: 537788
                - NK cells found: 537788
                - Percentage of NK cells: 100.00%
                
Successfully saved processed file as /app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/nk_cells.h5ad


In [3]:
adata_test= sc.read_h5ad('/app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/nk_cells.h5ad')
adata_test.var

,soma_joinid,feature_name,feature_length,nnz,n_measured_obs,original_names,gene_symbols
TSPAN6,0,TSPAN6,4530,4530448,73855064,ENSG00000000003,TSPAN6
TNMD,1,TNMD,1476,236059,61201828,ENSG00000000005,TNMD
DPM1,2,DPM1,9276,17576462,74159149,ENSG00000000419,DPM1
SCYL3,3,SCYL3,6883,9117322,73988868,ENSG00000000457,SCYL3
FIRRM,4,C1orf112,5970,6287794,73636201,ENSG00000000460,FIRRM
...,...,...,...,...,...,...,...
ENSG00000288718,60525,ENSG00000288718.1,1070,4,1248980,ENSG00000288718,None_18342
ENSG00000288719,60526,ENSG00000288719.1,4252,2826,1248980,ENSG00000288719,None_18343
ENSG00000288724,60527,ENSG00000288724.1,625,36,1248980,ENSG00000288724,None_18344
ENSG00000290791,60528,ENSG00000290791.1,3612,1642,43485,ENSG00000290791,None_18345


In [4]:
import scanpy as sc
import pandas as pd
import re
from typing import List, Optional, Dict, Set
import numpy as np

def is_valid_gene_symbol(symbol: str, common_symbols: Optional[Set[str]] = None) -> bool:
    """
    Verificación más estricta de símbolos de genes.
    """
    if pd.isna(symbol):
        return False
    
    symbol = str(symbol)
    
    # Patrones que definitivamente NO son símbolos de genes
    invalid_patterns = [
        r'^ENSG\d+',           # Ensembl IDs
        r'^ENST\d+',           # Ensembl transcript IDs
        r'^NM_\d+',            # RefSeq mRNA
        r'^NR_\d+',            # RefSeq non-coding RNA
        r'^XM_\d+',            # RefSeq predicted mRNA
        r'^\d+$',              # Solo números
        r'^chr\d+',            # Cromosomas
        r'^LOC\d+',            # IDs de ubicación
        r'^[0-9XY]+p\d+',      # Bandas cromosómicas
        r'^hsa-mir-',          # microRNAs
        r'^5S_rRNA',           # RNAs ribosomales
        r'^sno\w+',            # small nucleolar RNAs
        r'^bP-\d+',            # IDs de BAC
        r'^yR\w+',             # otros IDs no estándar
        r'^AC\d+\.\d+',        # IDs de clon
        r'^RP\d+-\d+',         # IDs de clon
        r'^CTD-\d+',           # IDs de clon
    ]
    
    for pattern in invalid_patterns:
        if re.match(pattern, symbol):
            return False
    
    valid_pattern = r'^[A-Z][A-Z0-9\-]{0,24}[A-Z0-9]$'
    is_valid_format = bool(re.match(valid_pattern, symbol))
    
    if common_symbols is not None:
        return is_valid_format and symbol in common_symbols
    
    return is_valid_format

def load_common_gene_symbols() -> Set[str]:
    """
    Carga un conjunto de símbolos de genes conocidos desde HGNC.
    En tu caso, podrías adaptar esto para cargar desde un archivo local
    o usar una lista predefinida.
    """
    # Aquí podrías cargar tu propia lista de símbolos válidos
    # Por ahora retornamos None para usar solo la validación por patrón
    return None

def standardize_gene_symbols(adata: sc.AnnData, common_symbols: Optional[Set[str]] = None) -> sc.AnnData:
    """
    Estandariza los símbolos de genes en el objeto AnnData, manteniendo la consistencia dimensional.
    """
    print("Iniciando estandarización de símbolos de genes...")
    
    # Crear una copia para no modificar el original
    adata = adata.copy()
    
    # Lista de posibles columnas con símbolos de genes
    possible_gene_columns = [
        'gene_symbol', 'feature_name', 'feature_names', 
        'gene_name', 'gene_names', 'name', 'names'
    ]
    
    # Encontrar la mejor columna de símbolos
    best_column = None
    best_score = 0
    
    for col in possible_gene_columns:
        if col in adata.var.columns:
            valid_symbols = sum(adata.var[col].apply(
                lambda x: is_valid_gene_symbol(x, common_symbols)
            ))
            score = valid_symbols / len(adata.var)
            
            if score > best_score:
                best_score = score
                best_column = col
    
    if best_column is None:
        raise ValueError("No se encontró ninguna columna válida con símbolos de genes")
    
    print(f"Se identificó '{best_column}' como la mejor columna (score: {best_score:.2f})")
    
    # Crear nueva columna para símbolos estandarizados
    adata.var['standard_gene_symbol'] = adata.var[best_column].astype(str)
    
    # Mantener registro de genes válidos e inválidos
    valid_mask = adata.var['standard_gene_symbol'].apply(
        lambda x: is_valid_gene_symbol(x, common_symbols)
    )
    
    # Filtrar solo los genes válidos
    adata = adata[:, valid_mask].copy()
    
    # Manejar duplicados
    duplicates = adata.var['standard_gene_symbol'].duplicated()
    if duplicates.any():
        n_duplicates = duplicates.sum()
        print(f"Se encontraron {n_duplicates} símbolos duplicados")
        
        # Mantener solo la primera ocurrencia de cada gen duplicado
        adata = adata[:, ~duplicates].copy()
    
    # Establecer los símbolos estandarizados como índice
    adata.var.set_index('standard_gene_symbol', inplace=True)
    
    print(f"Dimensiones finales: {adata.shape}")
    return adata

def fix_integrated_gene_symbols(adata: sc.AnnData) -> sc.AnnData:
    """
    Corrige los símbolos de genes en un dataset ya integrado.
    """
    print("Iniciando corrección de símbolos de genes...")
    
    # Cargar símbolos comunes si están disponibles
    common_symbols = load_common_gene_symbols()
    
    # Aplicar estandarización
    corrected = standardize_gene_symbols(adata, common_symbols)
    
    print("Corrección completada")
    return corrected

# Ejemplo de uso:
def process_and_save_data(input_path: str, output_path: str):
    """
    Procesa y guarda los datos con manejo de errores.
    """
    try:
        # Cargar dataset
        print(f"Cargando datos desde {input_path}")
        adata = sc.read_h5ad(input_path)
        
        # Aplicar corrección
        adata_corrected = fix_integrated_gene_symbols(adata)
        
        # Guardar resultado
        print(f"Guardando datos en {output_path}")
        adata_corrected.write(output_path)
        
        # Verificar que se puede cargar el archivo guardado
        print("Verificando archivo guardado...")
        test_load = sc.read_h5ad(output_path)
        print(f"Verificación exitosa. Dimensiones finales: {test_load.shape}")
        
    except Exception as e:
        print(f"Error durante el procesamiento: {str(e)}")
        raise

input_path = '/app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/nk_cells.h5ad'
output_path = '/app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/test/nk_corrected_test.h5ad'

process_and_save_data(input_path, output_path)

Cargando datos desde /app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/nk_cells.h5ad
Iniciando corrección de símbolos de genes...
Iniciando estandarización de símbolos de genes...
Se identificó 'feature_name' como la mejor columna (score: 0.66)
Dimensiones finales: (537788, 40166)
Corrección completada
Guardando datos en /app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/test/nk_corrected_test.h5ad
Verificando archivo guardado...
Verificación exitosa. Dimensiones finales: (537788, 40166)


In [6]:
import scanpy as sc
direccion= '/app/project/restore_data/pipeline_articulo/h5ad/nk_full_0325/nk_final/nk_final_final/test/nk_corrected_test.h5ad'
adata = sc.read_h5ad(direccion)
adata.var

,soma_joinid,feature_name,feature_length,nnz,n_measured_obs,original_names,gene_symbols
standard_gene_symbol,,,,,,,
TSPAN6,0,TSPAN6,4530,4530448,73855064,ENSG00000000003,TSPAN6
TNMD,1,TNMD,1476,236059,61201828,ENSG00000000005,TNMD
DPM1,2,DPM1,9276,17576462,74159149,ENSG00000000419,DPM1
SCYL3,3,SCYL3,6883,9117322,73988868,ENSG00000000457,SCYL3
FGR,5,FGR,3382,5667858,74061500,ENSG00000000938,FGR
...,...,...,...,...,...,...,...
DUS4L-BCAP29,60459,DUS4L-BCAP29,7268,31561,5301281,ENSG00000288558,DUS4L-BCAP29
OR4F8BP,60467,OR4F8BP,938,21,841693,ENSG00000286255,OR4F8BP
OR7E11P,60470,OR7E11P,952,975,2596589,ENSG00000285537,OR7E11P


In [7]:
# Usando re para expresiones regulares case insensitive
import re 

def verificar_patrones_genes(adata):
    # Buscar genes que empiezan con 'none' (case insensitive)
    patron_none = re.compile(r'^none', re.IGNORECASE)
    none_count = sum(adata.var.index.str.match(patron_none))
    
    # Buscar genes que empiezan con 'ENSG'
    patron_ensg = re.compile(r'^ENSG')
    ensg_count = sum(adata.var.index.str.match(patron_ensg))
    
    # Obtener los nombres específicos de los genes
    genes_none = adata.var.index[adata.var.index.str.match(patron_none)].tolist()
    genes_ensg = adata.var.index[adata.var.index.str.match(patron_ensg)].tolist()
    
    print(f"Número de genes que empiezan con 'none' (case insensitive): {none_count}")
    print(f"Número de genes que empiezan con 'ENSG': {ensg_count}")
    
    if none_count > 0:
        print("\nGenes que empiezan con 'none':")
        for gene in genes_none:
            print(f"- {gene}")
            
    if ensg_count > 0:
        print("\nGenes que empiezan con 'ENSG':")
        for gene in genes_ensg:
            print(f"- {gene}")
    
    return {
        'none_genes': genes_none,
        'ensg_genes': genes_ensg,
        'none_count': none_count,
        'ensg_count': ensg_count
    }

# Para usar la función:
resultados = verificar_patrones_genes(adata)

Número de genes que empiezan con 'none' (case insensitive): 0
Número de genes que empiezan con 'ENSG': 0


In [8]:
adata

AnnData object with n_obs × n_vars = 537788 × 40166
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'cell_group', 'age_yrs', 'age_group'
    var: 'soma_joinid', 'feature_name', 'feature_length', 'nnz', 'n_measured_obs', 'original_names', 'gene_symbols'

In [9]:
adata.obs['dataset_id'].value_counts()

dataset_id
3faad104-2ab8-4434-816d-474d8d2641db    171939
b0e547f0-462b-4f81-b31b-5b0a5d96f537    147484
2c820d53-cbd7-4e0a-be7a-a0ad1989a98f     48055
218acb0f-9f2f-4f76-b90b-15a4b7c7f629     38304
2a498ace-872a-4935-984b-1afa70fd9886     25246
c7775e88-49bf-4ba2-a03b-93f00447c958     19755
242c6e7f-9016-4048-af70-d631f5eea188     19277
ebc2e1ff-c8f9-466a-acf4-9d291afaf8b3     11952
9dbab10c-118d-496b-966a-67f1763a6b7d      8557
30cd5311-6c09-46c9-94f1-71fe4b91813c      8307
5af90777-6760-4003-9dba-8f945fec6fdf      7591
456e8b9b-f872-488b-871d-94534090a865      4707
59b69042-47c2-47fd-ad03-d21beb99818f      4693
3c75a463-6a87-4132-83a8-c3002624394d      4484
1b9d8702-5af8-4142-85ed-020eb06ec4f6      3984
de2c780c-1747-40bd-9ccf-9588ec186cee      3915
53d208b0-2cfd-4366-9866-c3c6114081bc      3466
19e46756-9100-4e01-8b0e-23b557558a4c      2721
d7d7e89c-c93a-422d-8958-9b4a90b69558      1283
5e717147-0f75-4de1-8bd2-6fda01b8d75f       968
8c42cfd0-0b0a-46d5-910c-fc833d83c45e       519
d3

In [10]:
adata.obs['age_group'].value_counts()

age_group
old      187173
adult    177802
young    172813
Name: count, dtype: int64

In [ ]:
age_group
young    162334
adult    145073
old       58442
Name: count, dtype: int64

In [11]:
adata.obs['age_yrs'].value_counts()

age_yrs
non_specific    73781
20              48708
33              10596
34              10301
37              10273
                ...  
90                664
94                303
95                180
96                 81
97                 44
Name: count, Length: 83, dtype: int64

In [12]:
adata.obs['cell_type'].value_counts()

cell_type
natural killer cell                                      344293
CD16-positive, CD56-dim natural killer cell, human       164351
CD16-negative, CD56-bright natural killer cell, human     20677
mature NK T cell                                           7039
type I NK T cell                                           1194
activated type II NK T cell                                 234
Name: count, dtype: int64

In [13]:
# Primero veamos las categorías originales
print("Categorías originales:")
print(adata.obs['development_stage'].unique())
print("\nConteo de categorías originales:")
print(adata.obs['development_stage'].value_counts())

# Crear máscara para filtrar los valores que contengan 'year-old human stage'
mask = ~adata.obs['development_stage'].str.contains('year-old human stage', na=False)

# Aplicar el filtro al AnnData
adata_filtered = adata[mask].copy()

# Ver las categorías restantes
print("\nCategorías después del filtrado:")
print(adata_filtered.obs['development_stage'].unique())
print("\nConteo de categorías después del filtrado:")
print(adata_filtered.obs['development_stage'].value_counts())

# Mostrar cuántas observaciones se filtraron
print(f"\nNúmero de observaciones originales: {adata.n_obs}")
print(f"Número de observaciones después del filtrado: {adata_filtered.n_obs}")
print(f"Observaciones removidas: {adata.n_obs - adata_filtered.n_obs}")

Categorías originales:
['41-year-old human stage', '26-year-old human stage', '75-year-old human stage', '49-year-old human stage', '36-year-old human stage', ..., '7-year-old human stage', '5-year-old human stage', '4-year-old human stage', '6-year-old human stage', 'third decade human stage']
Length: 96
Categories (96, object): ['19-year-old human stage', '2-5 year-old child stage', '20-year-old human stage', '21-year-old human stage', ..., 'newborn human stage', 'seventh decade human stage', 'sixth decade human stage', 'third decade human stage']

Conteo de categorías originales:
development_stage
20-year-old human stage      48708
fourth decade human stage    21031
sixth decade human stage     12810
33-year-old human stage      10596
34-year-old human stage      10301
                             ...  
90-year-old human stage        664
94-year-old human stage        303
95-year-old human stage        180
96-year-old human stage         81
97-year-old human stage         44
Name: c

In [14]:
# Crear un filtro para las observaciones con 'other' en age_group
mask_other = adata.obs['age_group'] == 'other'

# Ver los valores únicos de development_stage para estas observaciones
print("Valores únicos de development_stage para age_group 'other':")
print(adata.obs.loc[mask_other, 'development_stage'].unique())

# Ver el conteo detallado
print("\nConteo de cada valor de development_stage para age_group 'other':")
print(adata.obs.loc[mask_other, 'development_stage'].value_counts())

# Mostrar el número total de observaciones con age_group 'other'
print(f"\nTotal de observaciones con age_group 'other': {mask_other.sum()}")

Valores únicos de development_stage para age_group 'other':
[], Categories (96, object): ['19-year-old human stage', '2-5 year-old child stage', '20-year-old human stage', '21-year-old human stage', ..., 'newborn human stage', 'seventh decade human stage', 'sixth decade human stage', 'third decade human stage']

Conteo de cada valor de development_stage para age_group 'other':
development_stage
19-year-old human stage     0
2-5 year-old child stage    0
82-year-old human stage     0
81-year-old human stage     0
80-year-old human stage     0
                           ..
46-year-old human stage     0
45-year-old human stage     0
44-year-old human stage     0
43-year-old human stage     0
third decade human stage    0
Name: count, Length: 96, dtype: int64

Total de observaciones con age_group 'other': 0
